# Predicting requests until overwrite

LightGBM predicts the number of read and write requests until the next write at the same starting byte offset. Training and MAE evaluation use observed overwrites only.

In [1]:
import json
from pathlib import Path
from predict import FEATURES
from count_mae_control import predict_records

output = Path('mae_control')
results = {t: json.loads((output / f'{t}.json').read_text()) for t in ('prxy_1', 'prn_1')}

## Results

Extended evaluation; methods use the same observed outcomes. MAE is measured in requests, lower is better.

In [2]:
for trace, result in results.items():
    methods = result['windows']['evaluation_extended']['methods']
    print(trace, 'coverage:', f"{methods['direct_mae']['observed_fraction']:.1%}")
    for name in ('direct_mae', 'previous_gap', 'median_three_gaps', 'training_median'):
        print(' ', name, 'MAE:', round(methods[name]['mae']))

prxy_1 coverage: 98.2%
  direct_mae MAE: 3011
  previous_gap MAE: 7270
  median_three_gaps MAE: 4501
  training_median MAE: 27656
prn_1 coverage: 62.3%
  direct_mae MAE: 84317
  previous_gap MAE: 267691
  median_three_gaps MAE: 187085
  training_median MAE: 105369


## Predictions on recorded examples

In [3]:
for trace, result in results.items():
    examples = result['examples']
    records = [dict(zip(FEATURES, x['integer_features'])) for x in examples]
    rows = predict_records(trace, records, output)['rows']
    for example, row in zip(examples, rows):
        assert row['predicted_request_count'] == example['predictions']['direct_mae']
        print(trace, 'event:', example['event_index'], 'actual:', example['actual'], 'predicted:', row['predicted_request_count'])

prxy_1 event: 10000001 actual: 21620 predicted: 18897
prxy_1 event: 10000002 actual: 21620 predicted: 19142
prxy_1 event: 10000012 actual: 1267 predicted: 6181
prxy_1 event: 10000014 actual: 1263 predicted: 5860
prxy_1 event: 10000026 actual: 21652 predicted: 18812
prn_1 event: 10000000 actual: 20754 predicted: 21898
prn_1 event: 10000001 actual: 20657 predicted: 16215
prn_1 event: 10000002 actual: 21021 predicted: 33947
prn_1 event: 10000003 actual: 853 predicted: 52813
prn_1 event: 10000004 actual: 424 predicted: 21539


## Limitation

Unknown outcomes are excluded. On extended evaluation, 98.2% of prxy_1 and 62.3% of prn_1 outcomes are known. Some distance groups still have large errors; new traces have not been tested.